<a href="https://colab.research.google.com/github/GhostMachine-ai/Claude-Skills/blob/main/My_First_MCP_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First, install the SDK: pip install mcp
import asyncio
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent
import json
import os

# 1. Initialize the MCP Server
app = Server("ghostmachine-local-tools")

# 2. Define the Tools we are exposing to Claude
@app.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="generate_slack_gif",
            description="Generates an animated text GIF for Slack. Provide text, animation style, and color.",
            inputSchema={
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to animate"},
                    "style": {"type": "string", "enum": ["bounce", "explode", "zoom"]},
                    "color_hex": {"type": "string", "description": "Hex color code, e.g. #FF5733"}
                },
                "required": ["text", "style"]
            }
        ),
        Tool(
            name="list_local_files",
            description="Lists all files in the current directory so Claude can see what documents exist.",
            inputSchema={
                "type": "object",
                "properties": {}
            }
        )
    ]

# 3. Define what happens when Claude calls a tool
@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    if name == "generate_slack_gif":
        text = arguments.get("text")
        style = arguments.get("style")
        color = arguments.get("color_hex", "#FFFFFF")

        # In reality, this would trigger the 'slack-gif-creator/core/gif_builder.py' script
        # For now, we simulate the success response
        simulation_msg = f"Successfully generated '{style}' GIF with text '{text}' in color {color}. Saved to /tmp/slack_output.gif"

        return [TextContent(type="text", text=simulation_msg)]

    elif name == "list_local_files":
        files = os.listdir(".")
        return [TextContent(type="text", text=json.dumps(files, indent=2))]

    else:
        raise ValueError(f"Unknown tool: {name}")

# 4. Start the server loop via Standard Input/Output
async def main():
    # This allows Claude Desktop or the Claude CLI to communicate with this script
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop